IMPORTS

In [ ]:
from pathlib import Path
import geopandas as gpd
import rasterio
import numpy as np
from shapely.geometry import box
from rasterio.mask import mask
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import sys 
lib_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/robyns_libraries")
sys.path.append(str(lib_dir))
print(sys.path)
import Robyn_river_floods


In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_dir = base_path / "dphil_paper_2/results"

In [ ]:
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

In [ ]:
catchments_unionized_final = base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg"
catchments = gpd.read_file(catchments_unionized_final)
catchments.head()
# print("Catchments:", len(catchments))

In [ ]:
baseline_fluvial_ead_max = base_path / "dphil_paper_2/processed_data/nbs_river_catchment/baseline_fluvial_ead/baseline__fluvial__ead_max.tif"

In [ ]:
# Sum all valid pixels in the raster (J$)
with rasterio.open(baseline_fluvial_ead_max) as src:
    band = src.read(1, masked=True).astype("float64")
    # apply scale/offset if present
    scale  = (src.scales[0]  if getattr(src, "scales", None)  else 1.0) or 1.0
    offset = (src.offsets[0] if getattr(src, "offsets", None) else 0.0) or 0.0
    band = band * scale + offset
    band = np.ma.masked_invalid(band)

    baseline_fluvial_ead_total_J = float(band.sum())

baseline_fluvial_ead_total_USD = baseline_fluvial_ead_total_J / 150

print(f"Total baseline fluvial ead total (J$): {baseline_fluvial_ead_total_J:,.0f}")
print(f"Total baseline fluvial ead total (USD @150): ${baseline_fluvial_ead_total_USD:,.0f}")


In [ ]:
# 1) Zonal sum helper (sums raster values within each polygon; no area calc)
def zonal_sum_only(raster_path, gdf, id_col="catchment_uid", all_touched=False):
    rows = []
    with rasterio.open(raster_path) as src:
        gdf_proj = gdf.to_crs(src.crs).copy()
        gdf_proj["geometry"] = gdf_proj.geometry.buffer(0)  # repair invalid geoms
        rb = box(*src.bounds)
        gdf_proj = gdf_proj[gdf_proj.intersects(rb)].copy()

        nd = src.nodata
        scale = (src.scales[0] if getattr(src, "scales", None) else 1.0) or 1.0
        offset = (src.offsets[0] if getattr(src, "offsets", None) else 0.0) or 0.0

        for _, r in gdf_proj.iterrows():
            try:
                data, _ = mask(src, [r.geometry.__geo_interface__],
                               crop=True, filled=False, all_touched=all_touched)
            except ValueError:
                rows.append({id_col: r[id_col], "sum": 0.0})
                continue

            band = data[0].astype("float64")
            band = band * scale + offset  # apply scale/offset if present

            ma = np.ma.array(band, mask=np.ma.getmaskarray(band))  # keep outside masked
            if nd is not None:
                ma = np.ma.masked_where(band == nd, ma)
            ma = np.ma.masked_invalid(ma)

            rows.append({id_col: r[id_col], "sum": float(ma.sum()) if ma.count() else 0.0})

    out = pd.DataFrame(rows)
    # ensure every ID appears (zeros for non-overlapping polygons)
    all_ids = gdf[[id_col]].copy()
    out = all_ids.merge(out, on=id_col, how="left").fillna({"sum": 0.0})
    return out

In [ ]:
# 2) Compute avoided_ead sums (min/max) by numeric catchment_uid

zs_max = zonal_sum_only(baseline_fluvial_ead_max, catchments, id_col="catchment_uid") \
           .rename(columns={"sum": "baseline_ead_max"})

result = (catchments.merge(zs_max, on="catchment_uid"))

zs_max

out_path = output_dir / "flood_damage_results/catchment_baseline_ead_max_results.csv"
zs_max.to_csv(out_path, index=False)
out_path

In [ ]:
avoided_EAD_path = base_path / "dphil_paper_2/results/flood_damage_results/expected_annual_damages_catchment/avoided_EAD_by_catchment.csv"
avoided_EAD = pd.read_csv(avoided_EAD_path, dtype={"catchment_uid": "Int64"})  # keep IDs numeric if possible
avoided_EAD

In [ ]:
# MAX

# Align ID dtype
result["catchment_uid"] = pd.to_numeric(result["catchment_uid"], errors="coerce").astype("Int64")
avoided_EAD["catchment_uid"] = pd.to_numeric(avoided_EAD["catchment_uid"], errors="coerce").astype("Int64")

# Keep only needed columns
a = result[["catchment_uid", "baseline_ead_max"]].copy()
b = avoided_EAD[["catchment_uid", "avoided_ead_max"]].copy()

# Merge
cmp = a.merge(b, on="catchment_uid", how="left")

# Differences
cmp["difference_max"] = cmp["baseline_ead_max"] - cmp["avoided_ead_max"]
cmp["pct_difference_max"] = np.where(
    cmp["baseline_ead_max"] > 0,
    100.0 * cmp["difference_max"] / cmp["baseline_ead_max"],
    np.nan
)

# (Optional) avoided share: avoided / baseline (%)
cmp["avoided_share_pct"] = np.where(
    cmp["baseline_ead_max"] > 0,
    100.0 * cmp["avoided_ead_max"] / cmp["baseline_ead_max"],
    np.nan
)

# Peek
print(
    cmp.head(10).to_string(index=False, 
        formatters={
            "baseline_ead_max": "{:,.2f}".format,
            "avoided_ead_max": "{:,.2f}".format,
            "difference_max": "{:,.2f}".format,
            "pct_difference_max": "{:.1f}".format,
            "avoided_share_pct": "{:.1f}".format,
        }
    )
)

In [ ]:
# Optional: tidy + round a bit for readability
cmp_export = cmp.copy()
cmp_export["baseline_ead_max"] = cmp_export["baseline_ead_max"].round(2)
cmp_export["avoided_ead_max"]  = cmp_export["avoided_ead_max"].round(2)
cmp_export["difference_max"]   = cmp_export["difference_max"].round(2)
cmp_export["avoided_share_pct"]   = cmp_export["avoided_share_pct"].round(1)
cmp_export["pct_difference_max"]  = cmp_export["pct_difference_max"].round(1)

# Column order (adjust if you like)
cols = [
    "catchment_uid",
    "baseline_ead_max",
    "avoided_ead_max",
    "difference_max",
    "avoided_share_pct",
    "pct_difference_max",
]
cmp_export = cmp_export[[c for c in cols if c in cmp_export.columns]]

# Save
out_csv = output_dir / "flood_damage_results/expected_annual_damages_catchment/baseline_vs_avoided_ead_max_by_catchment.csv"
cmp_export.to_csv(out_csv, index=False)
print("Wrote:", out_csv)

MAPPING

In [ ]:
# === Map: avoided_share_pct by catchment (choropleth with labels) ============

# 1) Build avoided_share_pct from your two tables (no geometry here)
a = result[["catchment_uid", "baseline_ead_max"]].copy()
b = avoided_EAD[["catchment_uid", "avoided_ead_max"]].copy()

a["catchment_uid"] = pd.to_numeric(a["catchment_uid"], errors="coerce").astype("Int64")
b["catchment_uid"] = pd.to_numeric(b["catchment_uid"], errors="coerce").astype("Int64")

df = a.merge(b, on="catchment_uid", how="left")
base = df["baseline_ead_max"].astype(float)
avo  = df["avoided_ead_max"].astype(float)

share = np.where(base > 0, 100.0 * avo / base, np.nan)
share = np.where((base == 0) & (avo == 0), 0.0, share)  # both-zero -> 0%
df["avoided_share_pct"] = np.clip(share, 0, 100)

# 2) Join to catchments geometry
g = catchments.merge(df[["catchment_uid", "avoided_share_pct"]],
                     on="catchment_uid", how="left")

# Use a metric CRS for plotting/scale; EPSG:3448 for Jamaica if current is geographic
plot_g = g.to_crs("EPSG:3448") if getattr(g.crs, "is_geographic", False) else g

# 3) Plot
with mpl.rc_context(globals().get("NATURE_RC", {})):
    TITLE_FS = mpl.rcParams.get("axes.titlesize", 7)
    LABEL_FS = mpl.rcParams.get("axes.labelsize", 6)
    TICK_FS  = mpl.rcParams.get("xtick.labelsize", 5.5)

    fig, ax = plt.subplots(figsize=(Robyn_river_floods.mm_to_in(140), Robyn_river_floods.mm_to_in(70)))
    ax.set_axis_off()
    ax.set_aspect("equal")

    vmin, vmax = 0.0, 100.0
    cmap = mpl.colormaps["Blues"].with_extremes(bad=(0.92, 0.92, 0.92, 1.0))  # NaN = light grey

    # Choropleth
    plot_g.plot(
        column="avoided_share_pct",
        ax=ax, cmap=cmap, vmin=vmin, vmax=vmax,
        linewidth=0.35, edgecolor="white"
    )

    # Optional outline if you have jamaica_boundary
    try:
        jb = jamaica_boundary.to_crs(plot_g.crs)
        jb.boundary.plot(ax=ax, color="black", linewidth=0.6, zorder=5)
    except Exception:
        pass

    # Labels (UIDs) with white halo for legibility
    pts = plot_g.representative_point()
    for uid, x, y in zip(plot_g["catchment_uid"], pts.x, pts.y):
        if pd.isna(uid): 
            continue
        ax.text(x, y, str(int(uid)), ha="center", va="center", fontsize=6,
                color="black", path_effects=[pe.withStroke(linewidth=1.0, foreground="white")])

    # Colorbar
    sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax), cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.012)
    cbar.set_label("Avoided share of baseline EAD (%)", fontsize=LABEL_FS)
    cbar.set_ticks([0, 20, 40, 60, 80, 100])
    cbar.ax.tick_params(labelsize=TICK_FS, width=0.35, length=2)
    cbar.outline.set_linewidth(0.35)

    ax.set_title("Avoided share (%) by catchment", fontsize=TITLE_FS, pad=6)

    # Save
    out_csv = output_dir / "flood_damage_results/expected_annual_damages_catchment/baseline_vs_avoided_ead_max_by_catchment.csv"
    
    
    out_png = output_dir / "flood_damage_results/expected_annual_damages_catchment/avoided_share_pct_by_catchment_max_map.png"
    fig.savefig(out_png, dpi=600, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved:", out_png)